## LST

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from scipy.special import roots_legendre

import ipywidgets as widgets
from IPython.display import display, clear_output


# ------------------------------------------------------------
# Gauss-Legendre points on [0,1]
# ------------------------------------------------------------

def gauss_points(n):

    roots, weights = roots_legendre(n)

    pts = 0.5 * (roots + 1.0)

    wts = 0.5 * weights

    return pts, wts


# ------------------------------------------------------------
# Duffy map: square [0,1]^2 -> reference triangle
# ------------------------------------------------------------

def duffy_map(z_hat, e_hat):

    zeta   = z_hat

    eta    = e_hat * (1.0 - z_hat)

    weight = (1.0 - z_hat)

    return zeta, eta, weight


# ------------------------------------------------------------
# Plot function
# ------------------------------------------------------------

def plot_duffy(n, z_h, e_h):

    fig = plt.figure(figsize=(13, 6))

    gs  = gridspec.GridSpec(
        1, 2,
        left=0.07,
        right=0.96,
        top=0.88,
        bottom=0.16,
        wspace=0.35
    )

    ax_sq  = fig.add_subplot(gs[0])

    ax_tri = fig.add_subplot(gs[1])

    # --------------------------------------------------------
    # Square
    # --------------------------------------------------------

    ax_sq.set_title('Unit square [0,1]² — (ζ̂, η̂)', fontsize=11)

    ax_sq.set_xlim(-0.05, 1.05)

    ax_sq.set_ylim(-0.05, 1.05)

    ax_sq.set_xlabel('ζ̂')

    ax_sq.set_ylabel('η̂')

    ax_sq.set_aspect('equal')

    ax_sq.add_patch(
        plt.Polygon(
            [[0, 0], [1, 0], [1, 1], [0, 1]],
            fill=True,
            facecolor='#e8f4f8',
            edgecolor='steelblue',
            lw=1.5
        )
    )

    # --------------------------------------------------------
    # Triangle
    # --------------------------------------------------------

    ax_tri.set_title('Reference triangle — (ζ, η)', fontsize=11)

    ax_tri.set_xlim(-0.05, 1.05)

    ax_tri.set_ylim(-0.05, 1.05)

    ax_tri.set_xlabel('ζ')

    ax_tri.set_ylabel('η')

    ax_tri.set_aspect('equal')

    ax_tri.add_patch(
        plt.Polygon(
            [[0, 0], [1, 0], [0, 1]],
            fill=True,
            facecolor='#fff3e0',
            edgecolor='darkorange',
            lw=1.5
        )
    )

    # --------------------------------------------------------
    # Gauss points
    # --------------------------------------------------------

    pts, wts = gauss_points(n)

    sq_x   = []

    sq_y   = []

    tri_x  = []

    tri_y  = []

    colors = []

    weights_display = []

    for zi, wi in zip(pts, wts):

        for ei, we in zip(pts, wts):

            zeta, eta, w_duffy = duffy_map(zi, ei)

            w_total = wi * we * w_duffy

            sq_x.append(zi)

            sq_y.append(ei)

            tri_x.append(zeta)

            tri_y.append(eta)

            colors.append(w_total)

            weights_display.append(w_total)

    colors = np.array(colors)

    if colors.max() > 0.0:

        c_norm = colors / colors.max()

    else:

        c_norm = colors

    ax_sq.scatter(
        sq_x,
        sq_y,
        s=70,
        zorder=5,
        c=c_norm,
        cmap='Blues',
        vmin=0,
        vmax=1
    )

    ax_tri.scatter(
        tri_x,
        tri_y,
        s=70,
        zorder=5,
        c=c_norm,
        cmap='Oranges',
        vmin=0,
        vmax=1
    )

    # --------------------------------------------------------
    # Highlighted point
    # --------------------------------------------------------

    z_sel, e_sel, w_sel = duffy_map(z_h, e_h)

    ax_sq.plot(
        [z_h],
        [e_h],
        'r*',
        ms=18,
        zorder=10,
        label='selected (ζ̂, η̂)'
    )

    ax_tri.plot(
        [z_sel],
        [e_sel],
        'r*',
        ms=18,
        zorder=10,
        label='mapped (ζ, η)'
    )

    ax_sq.legend(fontsize=8, loc='upper right')

    ax_tri.legend(fontsize=8, loc='upper right')

    # --------------------------------------------------------
    # Text info
    # --------------------------------------------------------

    fig.text(
        0.50,
        0.75,
        f'ζ̂ = {z_h:.3f},  η̂ = {e_h:.3f}\n'
        f'→  ζ = {z_sel:.3f},  η = {e_sel:.3f}\n'
        f'factor (1−ζ̂) = {w_sel:.3f}',
        ha='center',
        va='center',
        fontsize=10,
        bbox=dict(
            boxstyle='round,pad=0.4',
            fc='white',
            ec='gray',
            alpha=0.9
        )
    )

    fig.text(
        0.505,
        0.58,
        'ζ = ζ̂\nη = η̂(1−ζ̂)',
        ha='center',
        va='center',
        fontsize=9,
        color='gray',
        bbox=dict(
            boxstyle='rarrow,pad=0.4',
            fc='#f5f5f5',
            ec='gray',
            alpha=0.8
        )
    )

    fig.suptitle(
        'Duffy transform — mapping [0,1]² → reference triangle',
        fontsize=12,
        fontweight='bold'
    )

    plt.show()


# ------------------------------------------------------------
# Interactive controls for Jupyter / VSCode
# ------------------------------------------------------------

out = widgets.Output()

s_n = widgets.IntSlider(
    value=2,
    min=1,
    max=5,
    step=1,
    description='Gauss pts n',
    continuous_update=False
)

s_zh = widgets.FloatSlider(
    value=0.3,
    min=0.0,
    max=1.0,
    step=0.01,
    description='ζ̂',
    continuous_update=False
)

s_eh = widgets.FloatSlider(
    value=0.4,
    min=0.0,
    max=1.0,
    step=0.01,
    description='η̂',
    continuous_update=False
)


def update(change=None):

    with out:

        clear_output(wait=True)

        plot_duffy(
            n=s_n.value,
            z_h=s_zh.value,
            e_h=s_eh.value
        )


s_n.observe(update, names='value')

s_zh.observe(update, names='value')

s_eh.observe(update, names='value')


display(widgets.VBox([s_n, s_zh, s_eh]))

display(out)

update()

Output()